# LoRA Fine-Tuning for Medical Purpose

LoRA (Low-Rank Adaptation) freezes the entire pre-trained model and injects small trainable rank-decomposition matrices into specific layers. Only these tiny matrices are trained.

## The Math
Instead of updating the full weight matrix W, LoRA does:

$$\Delta W = B \cdot A \quad \text{where } B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times d}$$

- **W** → frozen (never updated)  
- **A, B** → trained (tiny low-rank matrices)  
- **r** → rank hyperparameter (typically 4–64)

## What Changes After LoRA?

| Aspect | Base Model (Before) | LoRA Fine-tuned (After) |
|---|---|---|
| General knowledge | Broad, generic answers | Same broad knowledge preserved |
| Domain responses | Generic/off-topic | Specific to your data |
| Output format | Random structure | Follows your exact format |
| Task accuracy | ~60–70% on custom task | ~85–95% on custom task |
| Saved file size | 13–240GB | +30–100MB adapter only |

## Key Hyperparameters

| Parameter | What it does | Typical Values |
|---|---|---|
| `r` (rank) | Size of low-rank matrices | 4, 8, 16, 32, 64 |
| `lora_alpha` | Scaling factor (α/r) | 16, 32 (often = 2×r) |
| `lora_dropout` | Dropout on LoRA layers | 0.05 – 0.1 |
| `target_modules` | Which layers to apply LoRA | q_proj, v_proj, k_proj |

## Step 1 — Install Dependencies

In [ ]:
!pip install transformers peft datasets bitsandbytes accelerate trl torch

## Step 2 — Prepare Medical Dataset

We create a dataset of medical instruction-input-output pairs.  
**LoRA learns the FORMAT and STYLE from this data — not the patient names.**  
Patient names come from the prompt at inference time.

Training data structure:
```
### Instruction: (what to do)
### Input:       (patient data — provided at runtime)
### Response:    (expected structured output format)
```

In [ ]:
import json
import os

# Your medical training data — teach the model YOUR hospital's format
medical_data = [
    {
        "instruction": "Summarize the patient diagnosis",
        "input": "Patient: Alice, 32F. BP: 120/80. Sugar: 95 mg/dL. Temp: 98.6F. Complaint: routine checkup.",
        "output": """Patient: Alice | Age: 32 | Gender: Female
Vitals: BP: 120/80 (Normal) | Sugar: 95 mg/dL (Normal) | Temp: 98.6F (Normal)
Diagnosis: Healthy — No abnormalities found
Medication: None required
Follow-up: Annual checkup in 12 months"""
    },
    {
        "instruction": "Summarize the patient diagnosis",
        "input": "Patient: Bob, 58M. BP: 162/100. Sugar: 310 mg/dL. HbA1c: 9.2%. Complaint: fatigue, frequent urination.",
        "output": """Patient: Bob | Age: 58 | Gender: Male
Vitals: BP: 162/100 (⚠️ HIGH) | Sugar: 310 mg/dL (⚠️ HIGH) | HbA1c: 9.2% (⚠️ HIGH)
Diagnosis: Stage 2 Hypertension + Type 2 Diabetes
Medication: Amlodipine 5mg (once daily) | Metformin 500mg (twice daily)
Follow-up: 2 weeks — recheck BP and sugar levels"""
    },
    {
        "instruction": "Suggest medication",
        "input": "Patient has Type 2 Diabetes. HbA1c: 8.5%. No kidney issues. No allergies.",
        "output": """Recommended Medication Protocol:
First-line: Metformin 500mg twice daily with meals
Target HbA1c: < 7.0%
Monitoring: Check HbA1c every 3 months
Lifestyle: Low-carb diet, 30 min exercise daily
Warning signs: Hypoglycemia (sugar < 70) — reduce dose"""
    },
    {
        "instruction": "Identify risk factors",
        "input": "Patient: 65M, smoker, BMI 32, BP 145/92, family history of heart disease, sedentary lifestyle.",
        "output": """Risk Assessment:
🔴 HIGH RISK — Cardiovascular Event
Risk Factors Identified:
1. Age > 60 (non-modifiable)
2. Smoking (modifiable) — immediate cessation advised
3. Obesity — BMI 32 (modifiable) — target BMI < 25
4. Hypertension — BP 145/92 (modifiable) — medication required
5. Family history (non-modifiable)
6. Sedentary lifestyle (modifiable)
10-Year CVD Risk Score: ~28%
Action: Cardiology referral within 1 week"""
    },
    {
        "instruction": "Summarize the patient diagnosis",
        "input": "Patient: Mary, 45F. BP: 135/85. Cholesterol: 240 mg/dL. LDL: 160. Complaint: headache, dizziness.",
        "output": """Patient: Mary | Age: 45 | Gender: Female
Vitals: BP: 135/85 (⚠️ ELEVATED) | Cholesterol: 240 mg/dL (⚠️ HIGH) | LDL: 160 (⚠️ HIGH)
Diagnosis: Pre-hypertension + Hypercholesterolemia
Medication: Atorvastatin 10mg (once daily at night)
Lifestyle: DASH diet, reduce saturated fats, 30 min walking daily
Follow-up: 4 weeks — recheck lipid panel"""
    }
]

# Format into instruction template
def format_prompt(item):
    return f"""### Instruction:
{item['instruction']}

### Input:
{item['input']}

### Response:
{item['output']}"""

formatted = [{"text": format_prompt(item)} for item in medical_data]

# Save dataset
os.makedirs("data", exist_ok=True)
with open("data/medical_data.json", "w") as f:
    json.dump(formatted, f, indent=2)

print(f"✅ Dataset created: {len(formatted)} samples")
print("\nSample entry:\n")
print(formatted[0]["text"])

## Step 3 — Load Base Model with 4-bit Quantization (QLoRA)

We use **QLoRA** = LoRA + 4-bit quantization of the frozen base model.

| Approach | GPU RAM (7B model) | Training Time |
|---|---|---|
| Full Fine-tuning | ~56GB | 10 hours |
| LoRA (r=8) | ~18GB | 2 hours |
| QLoRA (4-bit + r=8) | ~6GB | 3 hours |

> **GPU Requirements**: Mistral 7B / LLaMA 3 8B fits on 6–10GB VRAM (RTX 3080/3090).  
> For `gpt-oss:120b` you need ~80GB VRAM (cloud GPUs like A100).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# ─── CONFIG ───────────────────────────────────────────────
BASE_MODEL = "meta-llama/Llama-3-8B-Instruct"  # change to "mistralai/Mistral-7B-Instruct-v0.3" if preferred
OUTPUT_DIR = "./medical-lora-adapter"
DATA_PATH  = "./data/medical_data.json"
MAX_SEQ_LEN = 512
# ──────────────────────────────────────────────────────────

# 4-bit quantization config (fits 7-8B model on ~6-8GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True        # extra compression
)

print("Loading base model in 4-bit (QLoRA)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",                    # auto GPU/CPU distribution
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)  # prepare for 4-bit training

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Base model loaded successfully")

## Step 4 — Attach LoRA Adapters

LoRA injects small trainable matrices into the attention layers.  
The base model weights stay **completely frozen**.

```
Transformer Attention Block:
├── q_proj  ← LoRA A+B matrices injected here  (Query)
├── k_proj  ← LoRA A+B matrices injected here  (Key)
├── v_proj  ← LoRA A+B matrices injected here  (Value)
└── o_proj  ← LoRA A+B matrices injected here  (Output)
```

Only `A` and `B` matrices are trained — **0.06% of total parameters**.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,                        # rank — higher = more learning capacity
    lora_alpha=32,               # scaling = alpha/r = 2 (recommended: 2x rank)
    target_modules=[             # which attention layers get LoRA matrices
        "q_proj",                # Query projection
        "k_proj",                # Key projection
        "v_proj",                # Value projection
        "o_proj",                # Output projection
    ],
    lora_dropout=0.05,           # dropout for regularization
    bias="none",
    task_type=TaskType.CAUSAL_LM # text generation task
)

model = get_peft_model(model, lora_config)

# Show how few parameters are actually being trained
model.print_trainable_parameters()
# Expected output:
# trainable params: ~8,388,608 || all params: ~8,030,261,248 || trainable%: ~0.10%

## Step 5 — Train the Model

We use `SFTTrainer` (Supervised Fine-Tuning Trainer) from the `trl` library.  
Only the LoRA adapter weights get updated — the base model stays frozen throughout.

In [ ]:
import json
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

# Load dataset
with open(DATA_PATH) as f:
    data = json.load(f)
dataset = Dataset.from_list(data)
print(f"Training samples: {len(dataset)}")

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,                  # more epochs = better learning on small data
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,       # effective batch size = 8
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_32bit",           # memory-efficient optimizer for QLoRA
    report_to="none"
)

# SFT Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",           # column name in dataset
    max_seq_length=MAX_SEQ_LEN,
    packing=False
)

print("Starting LoRA training...")
trainer.train()
print("✅ Training complete!")

## Step 6 — Save the LoRA Adapter

The adapter is only **~30MB** compared to the full 16GB model.  
You can share/swap adapters while reusing the same base model for different tasks.

In [ ]:
# Save ONLY the LoRA adapter weights (~30MB, not the full 16GB model!)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os
adapter_files = os.listdir(OUTPUT_DIR)
print(f"✅ Adapter saved to: {OUTPUT_DIR}")
print(f"Files saved: {adapter_files}")
# adapter_model.safetensors  ← this is your tiny LoRA adapter file

## Step 7 — Inference (Use the Fine-Tuned Model)

### How inference works:
- **Base model** = general intelligence (language, reasoning)
- **LoRA adapter** = medical domain knowledge + your hospital's format
- **Your prompt** = today's patient data (name, BP, sugar — provided by YOU at runtime)

```
YOUR PROMPT:  "Patient: John, 45M. BP: 155/95. Sugar: 290..."
                                    ↑
              "John" comes from HERE — not from LoRA training!
              LoRA only teaches the FORMAT of the response.
```

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL   = "meta-llama/Llama-3-8B-Instruct"
ADAPTER_PATH = "./medical-lora-adapter"

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Attach LoRA adapter on top of base model
model_inf = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# Merge adapter into weights — no extra overhead at inference time
model_inf = model_inf.merge_and_unload()
model_inf.eval()

tokenizer_inf = AutoTokenizer.from_pretrained(ADAPTER_PATH)

def diagnose(instruction: str, patient_data: str) -> str:
    """
    instruction  = what to do (e.g. "Summarize the patient diagnosis")
    patient_data = actual patient info YOU provide at runtime (name, vitals, etc.)
    """
    prompt = f"""### Instruction:
{instruction}

### Input:
{patient_data}

### Response:
"""
    inputs = tokenizer_inf(prompt, return_tensors="pt").to(model_inf.device)

    with torch.no_grad():
        outputs = model_inf.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,           # low temp = more deterministic (good for medical)
            do_sample=True,
            pad_token_id=tokenizer_inf.eos_token_id
        )

    # Decode only the newly generated tokens (not the input prompt)
    response = tokenizer_inf.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

print("✅ Model ready for inference")

## Step 8 — Test With New Patients

Pass **any new patient data** at runtime. The model applies the learned medical format to it.

In [ ]:
# ─── Test 1: Diagnosis for a new patient ──────────────────
# "John" is provided HERE in the prompt — LoRA never saw this patient
result1 = diagnose(
    instruction="Summarize the patient diagnosis",
    patient_data="Patient: John, 45M. BP: 155/95. Sugar: 290 mg/dL. HbA1c: 8.8%. Complaint: blurred vision, fatigue."
)
print("=" * 60)
print("TEST 1 — DIAGNOSIS:")
print(result1)

# ─── Test 2: Medication suggestion ────────────────────────
result2 = diagnose(
    instruction="Suggest medication",
    patient_data="Patient has Stage 1 Hypertension. BP: 138/88. Age 50, no diabetes, no kidney issues."
)
print("\n" + "=" * 60)
print("TEST 2 — MEDICATION SUGGESTION:")
print(result2)

# ─── Test 3: Risk factor analysis ─────────────────────────
result3 = diagnose(
    instruction="Identify risk factors",
    patient_data="Patient: Sarah, 55F. Smoker. BMI 29. BP 148/94. Mother had stroke at 60."
)
print("\n" + "=" * 60)
print("TEST 3 — RISK FACTORS:")
print(result3)

## Summary

| | Role | Example |
|---|---|---|
| **Base Model** | General language intelligence | LLaMA 3 8B — knows language, reasoning |
| **LoRA Adapter** | Teaches domain format & rules | Medical diagnosis format, medication rules |
| **Your Prompt** | Actual data for today | Patient John, 45M, BP 155/95 |

### Key Takeaways
- LoRA trains **0.06–0.1%** of total parameters — massive memory savings
- The adapter file is **~30MB** vs 16GB for the full model
- Patient names/data always come from **your prompt at runtime**
- LoRA teaches **how to respond** — not what specific patients look like
- One base model + **multiple LoRA adapters** = multiple specialized models